# Data Preparation — CRISP-DM §3

Implements the pipeline in `section3_preprocessing_plan.md` for the Amazon
Reviews 2023 cross-domain recommendation project (EMCDR over 8 viable
vertical pairs).

The pipeline is split into two layers so a new pair never re-runs the
expensive per-vertical work:

- **Stage A — per-vertical layer.** Loops over all 5 verticals (Books,
  Movies & TV, CDs & Vinyl, Video Games, Toys & Games). Per vertical:
  load → clean → 5-core → labels → seen sets → sampled negatives → text
  embeddings (cached) → category multi-hot (shared vocab) → popularity
  priors → within-vertical join → temporal split → ID remap →
  write `data/processed/{vertical}/`. These are also the single-vertical
  baseline inputs.
- **Stage B — pair bridge.** Loops over the 8 pairs that clear the 10k
  shared-user floor (`Books → Movies_and_TV`, `Books → Toys_and_Games`,
  `Movies_and_TV → Toys_and_Games`, `Movies_and_TV → CDs_and_Vinyl`,
  `Books → CDs_and_Vinyl`, `Movies_and_TV → Video_Games`,
  `Toys_and_Games → Video_Games`, `Books → Video_Games`). Per pair:
  intersect users, assemble matched source→target pairs, role splits,
  write `data/processed/pairs/{SOURCE}__{TARGET}/`. Never reads `data/raw/`.
- **Stage C — describe.** Emits `data/processed/DATASET_DESCRIPTION.md`
  + the JSON mirror — the consolidated §3 deliverable.

Section numbering below mirrors the CRISP-DM deliverables:

| § | Deliverable |
|---|---|
| 3.1 | Rationale for Inclusion / Exclusion |
| 3.2 | Data Cleaning Report |
| 3.3 | Derived Attributes + Generated Records |
| 3.4 | Merged Data |
| 3.5 | Reformatted Data |


## 0. Setup

In [1]:
from __future__ import annotations
import json, os, hashlib, random, shutil, time, warnings
from dataclasses import dataclass, field, asdict
from pathlib import Path
from collections import Counter, defaultdict
from typing import Dict, List, Set, Tuple, Optional

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.csv as pacsv
from scipy.sparse import csr_matrix
from tqdm.auto import tqdm

PROJECT_ROOT = Path('/Users/orimood/Desktop/homework/Amazon_CR')
RAW          = PROJECT_ROOT / 'data' / 'raw'
PROCESSED    = PROJECT_ROOT / 'data' / 'processed'
PAIRS_DIR    = PROCESSED / 'pairs'
EMBED_CACHE  = PROJECT_ROOT / 'data' / 'embed_cache'

PROCESSED.mkdir(parents=True, exist_ok=True)
PAIRS_DIR.mkdir(parents=True, exist_ok=True)
EMBED_CACHE.mkdir(parents=True, exist_ok=True)


### Config

The plan's top-of-script config, as a dataclass. **`embed_sample_size = 2000`
is the smoke-mode default** — every vertical encodes at most 2000 new texts
this run; remaining items get the placeholder embedding. Flip to `0` for the
full run (~30 min on MPS, idempotent via the SHA1 text cache).


In [2]:
@dataclass
class Config:
    # per-vertical layer — all 5
    VERTICALS: Tuple[str, ...] = (
        'Books', 'Movies_and_TV', 'CDs_and_Vinyl',
        'Video_Games', 'Toys_and_Games',
    )
    # 8 pairs above the ~10k shared-user floor, ranked larger -> smaller
    PAIRS: Tuple[Tuple[str, str], ...] = (
        ('Books',         'Movies_and_TV'),
        ('Books',         'Toys_and_Games'),
        ('Movies_and_TV', 'Toys_and_Games'),
        ('Movies_and_TV', 'CDs_and_Vinyl'),
        ('Books',         'CDs_and_Vinyl'),
        ('Movies_and_TV', 'Video_Games'),
        ('Toys_and_Games','Video_Games'),
        ('Books',         'Video_Games'),
    )
    ADD_REVERSE: bool = False     # set True for symmetric ablation (16 pairs)
    OVERLAP_FLOOR: int = 10_000   # pair viability gate

    # density filter
    k_core: int = 5
    k_core_overrides: Dict[str, int] = field(default_factory=dict)  # e.g. {'Books': 3}

    # rating -> training signal
    positive_threshold: int = 4
    drop_ratings: Tuple[int, ...] = (0, 3)
    explicit_negative: Tuple[int, ...] = (1, 2)
    neg_sample_ratio: int = 4
    explicit_neg_weight: float = 1.0

    # text embeddings
    embed_model: str = 'sentence-transformers/all-MiniLM-L6-v2'
    embed_dim: int = 384
    max_seq_length: int = 256
    embed_normalize: bool = True
    text_fields: Tuple[str, ...] = ('title', 'description', 'features')
    text_fallback: Tuple[str, ...] = ('title', 'store', 'details')
    embed_batch: int = 256
    embed_sample_size: Optional[int] = 2000  # smoke default; flip to 0 for full run
    embed_device: str = 'auto'

    # splits
    split: str = 'temporal'
    split_ratios: Tuple[float, float, float] = (0.8, 0.1, 0.1)
    seed: int = 42

CFG = Config()
np.random.seed(CFG.seed); random.seed(CFG.seed)

print(json.dumps(asdict(CFG), indent=2, default=str))
print(f'\nVerticals: {len(CFG.VERTICALS)}   Pairs: {len(CFG.PAIRS)}')
print(f'Smoke mode: embed_sample_size={CFG.embed_sample_size} '
      f'({"FULL RUN" if not CFG.embed_sample_size else f"cap {CFG.embed_sample_size}/vertical"})')


{
  "VERTICALS": [
    "Books",
    "Movies_and_TV",
    "CDs_and_Vinyl",
    "Video_Games",
    "Toys_and_Games"
  ],
  "PAIRS": [
    [
      "Books",
      "Movies_and_TV"
    ],
    [
      "Books",
      "Toys_and_Games"
    ],
    [
      "Movies_and_TV",
      "Toys_and_Games"
    ],
    [
      "Movies_and_TV",
      "CDs_and_Vinyl"
    ],
    [
      "Books",
      "CDs_and_Vinyl"
    ],
    [
      "Movies_and_TV",
      "Video_Games"
    ],
    [
      "Toys_and_Games",
      "Video_Games"
    ],
    [
      "Books",
      "Video_Games"
    ]
  ],
  "ADD_REVERSE": false,
  "OVERLAP_FLOOR": 10000,
  "k_core": 5,
  "k_core_overrides": {},
  "positive_threshold": 4,
  "drop_ratings": [
    0,
    3
  ],
  "explicit_negative": [
    1,
    2
  ],
  "neg_sample_ratio": 4,
  "explicit_neg_weight": 1.0,
  "embed_model": "sentence-transformers/all-MiniLM-L6-v2",
  "embed_dim": 384,
  "max_seq_length": 256,
  "embed_normalize": true,
  "text_fields": [
    "title",
    "description",

In [3]:
# In-memory state — populated per stage, keyed by vertical.
RATINGS_5CORE: Dict[str, pd.DataFrame] = {}     # 5-core filtered (post drop)
ITEMS:         Dict[str, pd.DataFrame] = {}     # per-vertical item table (meta)
LABELS:        Dict[str, pd.DataFrame] = {}     # 5-core w/ label col (no sampled neg)
SEEN:          Dict[str, Dict[str, set]] = {}   # user_id -> set[asin], from full ratings
NEG:           Dict[str, pd.DataFrame] = {}     # sampled negatives
SIGNAL:        Dict[str, pd.DataFrame] = {}     # combined: pos + explicit-neg + sampled-neg
EMB:           Dict[str, np.ndarray] = {}       # (n_items, 384) float32
CAT:           Dict[str, csr_matrix] = {}       # (n_items, |vocab|) multi-hot
POP:           Dict[str, pd.DataFrame] = {}     # popularity priors per item

# Stats containers — feed Stage C.
SELECT_STATS:    Dict[str, Dict] = {}
CLEAN_STATS:     Dict[str, Dict] = {}
CONSTRUCT_STATS: Dict[str, Dict] = {'category_vocab_size': None, 'by_vertical': {}}
INTEGRATE_STATS: Dict[str, Dict] = {}
FORMAT_STATS:    Dict[str, Dict] = {}
PAIR_STATS:      Dict[str, Dict] = {}  # keyed by "SRC__TGT"

CAT_VOCAB: Dict[str, int] = {}

def log_stage(name: str, **kv):
    print(f'[{name}]')
    for k, v in kv.items():
        if isinstance(v, dict):
            print(f'  {k}:')
            for kk, vv in v.items():
                print(f'    {kk}: {vv}')
        else:
            print(f'  {k}: {v}')


### Clean up old single-pair output directory

The previous pipeline wrote everything to `data/processed/Books__Movies_and_TV/`.
The new layout puts Stage A in `data/processed/{vertical}/` and Stage B in
`data/processed/pairs/{SRC}__{TGT}/`. Delete the old directory to avoid
confusion. The embedding cache is preserved.

In [4]:
OLD_PAIR_DIR = PROCESSED / 'Books__Movies_and_TV'
if OLD_PAIR_DIR.exists():
    sz_mb = sum(p.stat().st_size for p in OLD_PAIR_DIR.rglob('*') if p.is_file()) / 1e6
    print(f'removing legacy {OLD_PAIR_DIR.relative_to(PROJECT_ROOT)} ({sz_mb:,.1f} MB)')
    shutil.rmtree(OLD_PAIR_DIR)
else:
    print(f'no legacy {OLD_PAIR_DIR.relative_to(PROJECT_ROOT)} to remove')

# new layout
print(f'\nStage A outputs:  {PROCESSED.relative_to(PROJECT_ROOT)}/<vertical>/')
print(f'Stage B outputs:  {PAIRS_DIR.relative_to(PROJECT_ROOT)}/<SRC>__<TGT>/')


removing legacy data/processed/Books__Movies_and_TV (1,748.8 MB)



Stage A outputs:  data/processed/<vertical>/
Stage B outputs:  data/processed/pairs/<SRC>__<TGT>/


## §3.1 Select Data — *Rationale for Inclusion / Exclusion*

This is the CRISP-DM §3.1 deliverable. Selection is two-tiered:

1. **Per-vertical layer (Stage A) — keep all 5 verticals.** §2.3 showed 8 of
   10 pairs clear the ~10k shared-user floor; processing all 5 once makes
   every pair available with no rework.
2. **Pair-set (Stage B) — keep the 8 viable pairs.** Sub-floor pairs
   (Toys ↔ CDs ≈ 7.75k, CDs ↔ Video Games ≈ 4.4k) are excluded and flagged
   as **R1 contingency candidates** (relax to 3-core).

Column selection (from §2.1, confirmed by §2.2):

- Ratings: `user_id`, `parent_asin`, `rating`, `timestamp` (all four).
- Metadata kept: `parent_asin`, `title`, `description`, `features`,
  `categories`, `main_category`, `store`, `details`, `average_rating`,
  `rating_number`, `author` (Books only).
- Metadata excluded: `images`, `videos`, `price`, `subtitle`, and
  `bought_together` (100% null per §2.4 — universally dropped at §3.2).

Row selection: 5-core, applied per vertical independently, iterated to
stability. Implementation: code lives below; this section's narrative IS
the §3.1 deliverable.

In [5]:
def load_ratings(vertical: str) -> pd.DataFrame:
    p = RAW / 'reviews' / f'{vertical}.csv'
    tbl = pacsv.read_csv(
        p,
        convert_options=pacsv.ConvertOptions(column_types={
            'user_id':     'string',
            'parent_asin': 'string',
            'rating':      'float32',
            'timestamp':   'int64',
        })
    )
    df = tbl.to_pandas()
    df['rating'] = df['rating'].astype('int8')
    return df


def baseline_counts(df: pd.DataFrame) -> Dict:
    return {
        'interactions_0core': int(len(df)),
        'users_0core': int(df['user_id'].nunique()),
        'items_0core': int(df['parent_asin'].nunique()),
    }


def drop_invalid(df: pd.DataFrame, vertical: str) -> Tuple[pd.DataFrame, Dict]:
    n_drop_0 = int((df['rating'] == 0).sum())
    n_drop_3 = int((df['rating'] == 3).sum())
    df_out = df[~df['rating'].isin(CFG.drop_ratings)].reset_index(drop=True)
    print(f'  {vertical:18s} rating==0 dropped {n_drop_0:>10,d}   '
          f'rating==3 dropped {n_drop_3:>10,d}   remaining {len(df_out):>12,d}')
    return df_out, {
        'dropped_rating_0': n_drop_0,
        'dropped_rating_3': n_drop_3,
        'interactions_after_drop': int(len(df_out)),
    }


def kcore_filter(df: pd.DataFrame, k: int, max_iters: int = 30) -> pd.DataFrame:
    prev = -1
    for _ in range(max_iters):
        u = df.groupby('user_id').size()
        df = df[df['user_id'].isin(u[u >= k].index)]
        i = df.groupby('parent_asin').size()
        df = df[df['parent_asin'].isin(i[i >= k].index)]
        if len(df) == prev:
            return df.reset_index(drop=True)
        prev = len(df)
    return df.reset_index(drop=True)


def post_5core(df_5c: pd.DataFrame, baseline_0core: int, k: int) -> Dict:
    return {
        'k_core': int(k),
        'interactions_5core': int(len(df_5c)),
        'users_5core': int(df_5c['user_id'].nunique()),
        'items_5core': int(df_5c['parent_asin'].nunique()),
        'retention_pct_vs_0core': round(len(df_5c) / baseline_0core * 100, 2),
    }


In [6]:
# Stage A.1: load → drop_invalid → 5-core, per vertical.
# Keeps the post-drop full ratings table in memory ONLY long enough to build
# seen sets (§3.3); freed right after.
RAW_RATINGS: Dict[str, pd.DataFrame] = {}  # post-drop, pre-5core; for seen sets

for vert in CFG.VERTICALS:
    t0 = time.time()
    print(f'\n=== {vert} ===')
    df = load_ratings(vert)
    print(f'  loaded            rows={len(df):>12,d}   ({time.time()-t0:.1f}s)')
    SELECT_STATS[vert] = {'vertical': vert, **baseline_counts(df)}

    df_post_drop, drop_counts = drop_invalid(df, vert)
    SELECT_STATS[vert].update(drop_counts)
    RAW_RATINGS[vert] = df_post_drop  # used for SEEN[] then dropped

    t1 = time.time()
    k_v = CFG.k_core_overrides.get(vert, CFG.k_core)
    df_5c = kcore_filter(df_post_drop, k_v)
    print(f'  {vert:18s} 5-core: rows {len(df_post_drop):>12,d} -> {len(df_5c):>12,d}   '
          f'({time.time()-t1:.1f}s)')
    SELECT_STATS[vert].update(
        post_5core(df_5c, SELECT_STATS[vert]['interactions_0core'], k_v))
    RATINGS_5CORE[vert] = df_5c

log_stage('3.1_select', **{v: SELECT_STATS[v] for v in CFG.VERTICALS})



=== Books ===


  loaded            rows=  29,139,329   (7.3s)


  Books              rating==0 dropped          4   rating==3 dropped  2,032,688   remaining   27,106,637


  Books              5-core: rows   27,106,637 ->    8,130,341   (61.9s)



=== Movies_and_TV ===


...

  Video_Games        5-core: rows    4,220,244 ->      682,040   (5.9s)

=== Toys_and_Games ===


  loaded            rows=  16,052,440   (3.3s)


  Toys_and_Games     rating==0 dropped          0   rating==3 dropped  1,090,727   remaining   14,961,713


  Toys_and_Games     5-core: rows   14,961,713 ->    3,360,986   (29.9s)


[3.1_select]
  Books:
    vertical: Books
    interactions_0core: 29139329
    users_0core: 10297355
    items_0core: 4446065
    dropped_rating_0: 4
    dropped_rating_3: 2032688
    interactions_after_drop: 27106637
    k_core: 5
    interactions_5core: 8130341
    users_5core: 687023
    items_5core: 440134
    retention_pct_vs_0core: 27.9
  Movies_and_TV:
    vertical: Movies_and_TV
    interactions_0core: 17158519
    users_0core: 6503429
    items_0core: 747764
    dropped_rating_0: 0
    dropped_rating_3: 1248470
    interactions_after_drop: 15910049
    k_core: 5
    interactions_5core: 6486790
    users_5core: 592843
    items_5core: 181532
    retention_pct_vs_0core: 37.81
  CDs_and_Vinyl:
    vertical: CDs_and_Vinyl
    interactions_0core: 4772071
    users_0core: 1754118
    items_0core: 701673
    dropped_rating_0: 0
    dropped_rating_3: 280338
    interactions_after_drop: 4491733
    k_core: 5
    interactions_5core: 1370152
    users_5core: 112342
    items_5core: 80990

## §3.2 Clean Data — *Data Cleaning Report*

Maps every §2.4 finding to exactly one action (per the spec):

1. **Out-of-range ratings.** The 4 Books rows with `rating == 0` are dropped
   above in §3.1 along with the rating==3 neutrals (verified count reported).
2. **Universally-null field.** `bought_together` (100% null per §2.4) is
   simply never read by `stream_meta_for_kept` below.
3. **Missing rich text — fallback chain.** `build_text` tries
   `title + description + features` (primary); if empty, falls back to
   `title + store + details`; if still empty, the item is flagged
   `text_source='none'` and receives the shared placeholder embedding in
   §3.3 (rather than being dropped).
4. **Categorical normalization.** `_normalize` lowercases / collapses
   whitespace on `store` and each `categories` path element.
5. **`main_category` correctness.** Stamped only as a feature; the
   source-of-truth vertical comes from the file the record was loaded from
   (`source_vertical` field on each item row).
6. **Clean-axes assertions** (`assert_quality`): no null keys, no dup
   (user, item) pairs (latest-ts wins if any appear in a future refresh),
   timestamps within 1996–2024, zero orphan interactions after join.

In [7]:
def _normalize(s: Optional[str]) -> Optional[str]:
    if not isinstance(s, str):
        return None
    s = ' '.join(s.split())
    return s if s else None


def _join_list(xs) -> str:
    if not isinstance(xs, list):
        return ''
    return '\n'.join(x.strip() for x in xs if isinstance(x, str) and x.strip())


def _details_to_text(d) -> str:
    if not isinstance(d, dict):
        return ''
    parts = []
    for k, v in d.items():
        if isinstance(v, list):
            v_str = ', '.join(x for x in v if isinstance(x, str))
        elif isinstance(v, str):
            v_str = v
        else:
            v_str = str(v)
        if v_str:
            parts.append(f'{k}: {v_str}')
    return '. '.join(parts)


def build_text(rec: Dict,
               text_fields: Tuple[str, ...],
               text_fallback: Tuple[str, ...]) -> Tuple[str, str]:
    """Return (text, source) where source ∈ {'primary','fallback','none'}."""
    chunks = []
    if 'title' in text_fields and rec.get('title'):
        chunks.append(_normalize(rec['title']))
    if 'description' in text_fields:
        d = _join_list(rec.get('description'))
        if d: chunks.append(d)
    if 'features' in text_fields:
        f = _join_list(rec.get('features'))
        if f: chunks.append(f)
    primary = '\n'.join(c for c in chunks if c)
    if primary:
        return primary, 'primary'

    chunks = []
    if 'title' in text_fallback and rec.get('title'):
        chunks.append(_normalize(rec['title']))
    if 'store' in text_fallback and _normalize(rec.get('store')):
        chunks.append(_normalize(rec['store']))
    if 'details' in text_fallback:
        d = _details_to_text(rec.get('details'))
        if d: chunks.append(d)
    fallback = '. '.join(c for c in chunks if c)
    if fallback:
        return fallback, 'fallback'
    return '', 'none'


In [8]:
TRACKED_FIELDS = ['title', 'description', 'features', 'categories',
                  'main_category', 'store', 'details', 'price',
                  'average_rating', 'rating_number', 'images', 'author',
                  'bought_together']  # tracked to *prove* it's 100% null


def stream_meta_for_kept(vertical: str, kept_asins: Set[str]) -> Tuple[pd.DataFrame, Dict]:
    """One pass over data/raw/meta/meta_{vertical}.jsonl: build the item table
    for the kept ASINs AND tally per-field missing-value rates for §3.2."""
    p = RAW / 'meta' / f'meta_{vertical}.jsonl'
    rows = []
    n_total = n_kept = n_primary = n_fallback = n_none = 0
    populated = Counter()
    with open(p) as f:
        for line in tqdm(f, desc=f'meta {vertical}', unit=' rec'):
            n_total += 1
            r = json.loads(line)
            a = r.get('parent_asin')
            if a not in kept_asins:
                continue
            n_kept += 1
            for fld in TRACKED_FIELDS:
                v = r.get(fld)
                if v is None: continue
                if isinstance(v, (list, dict)) and len(v) == 0: continue
                if isinstance(v, str) and not v.strip(): continue
                populated[fld] += 1
            text, src_tag = build_text(r, CFG.text_fields, CFG.text_fallback)
            if   src_tag == 'primary':  n_primary  += 1
            elif src_tag == 'fallback': n_fallback += 1
            else:                       n_none     += 1
            cats = r.get('categories') or []
            try:
                avg_rating = float(r.get('average_rating')) if r.get('average_rating') is not None else np.nan
            except (TypeError, ValueError):
                avg_rating = np.nan
            try:
                rating_num = int(r.get('rating_number')) if r.get('rating_number') is not None else 0
            except (TypeError, ValueError):
                rating_num = 0
            rows.append({
                'parent_asin':    a,
                'source_vertical': vertical,   # spec §3.2: vertical = file of origin
                'item_text':      text,
                'has_text':       src_tag != 'none',
                'text_source':    src_tag,
                'categories':     [_normalize(c) for c in cats if isinstance(c, str)],
                'main_category':  _normalize(r.get('main_category')),
                'store_norm':     _normalize(r.get('store')),
                'average_rating': avg_rating,
                'rating_number':  rating_num,
            })
    items = pd.DataFrame(rows)
    stats = {
        'meta_streamed':         n_total,
        'items_kept':            n_kept,
        'text_primary':          n_primary,
        'text_fallback':         n_fallback,
        'text_none_placeholder': n_none,
        'missing_value_rates':   {k: round(1 - populated.get(k, 0) / max(1, n_kept), 4)
                                  for k in TRACKED_FIELDS},
    }
    return items, stats


def assert_quality(ratings: pd.DataFrame, items: pd.DataFrame, vertical: str
                   ) -> Tuple[pd.DataFrame, Dict]:
    assert ratings['user_id'].notna().all(),     f'{vertical}: null user_id'
    assert ratings['parent_asin'].notna().all(), f'{vertical}: null parent_asin'
    assert ratings['rating'].between(1, 5).all(),f'{vertical}: rating out of [1,5]'
    ts = pd.to_datetime(ratings['timestamp'], unit='ms')
    assert (ts >= pd.Timestamp('1996-01-01')).all() and \
           (ts <  pd.Timestamp('2024-01-01')).all(), \
           f'{vertical}: timestamps out of expected window'
    item_set = set(items['parent_asin'])
    n_orphan = int((~ratings['parent_asin'].isin(item_set)).sum())
    assert n_orphan == 0, f'{vertical}: {n_orphan} orphan interactions'
    n_dup = int(ratings.duplicated(subset=['user_id', 'parent_asin']).sum())
    if n_dup:
        print(f'  {vertical}: deduping {n_dup:,} duplicate pairs (latest-ts wins)')
        idx = ratings.sort_values('timestamp').groupby(['user_id', 'parent_asin']).tail(1).index
        ratings = ratings.loc[idx].reset_index(drop=True)
    return ratings, {'orphan_interactions': n_orphan, 'duplicate_pairs': n_dup}


In [9]:
# Stage A.2: meta stream + clean-axes assertion, per vertical.
for vert in CFG.VERTICALS:
    print(f'\n=== {vert} ===')
    kept_asins = set(RATINGS_5CORE[vert]['parent_asin'].unique())
    t0 = time.time()
    items, st = stream_meta_for_kept(vert, kept_asins)
    print(f'  kept {st["items_kept"]:>9,d}   '
          f'primary={st["text_primary"]:>9,d}   '
          f'fallback={st["text_fallback"]:>9,d}   '
          f'none={st["text_none_placeholder"]:>9,d}   '
          f'({time.time()-t0:.1f}s)')
    ratings, q = assert_quality(RATINGS_5CORE[vert], items, vert)
    RATINGS_5CORE[vert] = ratings
    ITEMS[vert] = items
    CLEAN_STATS[vert] = {**st, **q}

log_stage('3.2_clean', **{v: {k: CLEAN_STATS[v][k] for k in
                              ('items_kept', 'text_primary', 'text_fallback',
                               'text_none_placeholder', 'orphan_interactions',
                               'duplicate_pairs')}
                          for v in CFG.VERTICALS})



=== Books ===


meta Books: 0 rec [00:00, ? rec/s]

  kept   440,134   primary=  440,134   fallback=        0   none=        0   (53.1s)



=== Movies_and_TV ===


meta Movies_and_TV: 0 rec [00:00, ? rec/s]

...

meta Video_Games: 0 rec [00:00, ? rec/s]

  kept    22,746   primary=   22,746   fallback=        0   none=        0   (1.7s)

=== Toys_and_Games ===


meta Toys_and_Games: 0 rec [00:00, ? rec/s]

  kept   148,572   primary=  148,570   fallback=        2   none=        0   (12.0s)


[3.2_clean]
  Books:
    items_kept: 440134
    text_primary: 440134
    text_fallback: 0
    text_none_placeholder: 0
    orphan_interactions: 0
    duplicate_pairs: 0
  Movies_and_TV:
    items_kept: 181532
    text_primary: 106431
    text_fallback: 75097
    text_none_placeholder: 4
    orphan_interactions: 0
    duplicate_pairs: 0
  CDs_and_Vinyl:
    items_kept: 80990
    text_primary: 80989
    text_fallback: 1
    text_none_placeholder: 0
    orphan_interactions: 0
    duplicate_pairs: 0
  Video_Games:
    items_kept: 22746
    text_primary: 22746
    text_fallback: 0
    text_none_placeholder: 0
    orphan_interactions: 0
    duplicate_pairs: 0
  Toys_and_Games:
    items_kept: 148572
    text_primary: 148570
    text_fallback: 2
    text_none_placeholder: 0
    orphan_interactions: 0
    duplicate_pairs: 0


## §3.3 Construct Data — *Derived Attributes + Generated Records*

Builds every feature the modelling layer consumes:

- **Labels (`y`)** — rating ≥ 4 → 1, rating ∈ {1, 2} → 0, rating == 3 dropped.
- **Seen sets** — per user, the union of every interaction (incl. dropped 3s)
  used to exclude items from negative sampling.
- **Sampled negatives** (generated records) — seeded, uniform-over-unobserved,
  ratio `NEG_SAMPLE_RATIO=4`.
- **Text embeddings** — `all-MiniLM-L6-v2` (384-d), explicit truncation at
  256 tokens, L2-normalised, **cache shared across all 5 verticals** keyed by
  SHA1(text) so cross-vertical duplicates encode once.
- **Category multi-hot** — vocabulary built ONCE across all 5 item tables, so
  every vertical sits in the same category space.
- **Popularity priors** — `average_rating` pass-through; `rating_number` is
  log1p+z-scored per vertical.

### 3.3(a) Labels

In [10]:
def build_labels(df: pd.DataFrame, vertical: str) -> pd.DataFrame:
    df = df.copy()
    df['label'] = -1
    df.loc[df['rating'] >= CFG.positive_threshold, 'label'] = 1
    df.loc[df['rating'].isin(CFG.explicit_negative), 'label'] = 0
    assert (df['label'] != -1).all(), f'{vertical}: unlabelled rows'
    df['weight'] = np.where(df['label'] == 0, CFG.explicit_neg_weight, 1.0).astype('float32')
    return df


for vert in CFG.VERTICALS:
    LABELS[vert] = build_labels(RATINGS_5CORE[vert], vert)
    n_pos = int((LABELS[vert]['label'] == 1).sum())
    n_neg = int((LABELS[vert]['label'] == 0).sum())
    pos_ratio = n_pos / max(1, n_pos + n_neg)
    print(f'  {vert:18s} positives={n_pos:>10,d}   explicit_neg={n_neg:>10,d}   '
          f'pos_ratio={pos_ratio:.3f}')
    CONSTRUCT_STATS['by_vertical'].setdefault(vert, {})['label_distribution'] = {
        'positives': n_pos, 'explicit_negatives': n_neg,
        'dropped_3': int(SELECT_STATS[vert]['dropped_rating_3']),
    }


  Books              positives= 7,563,762   explicit_neg=   566,579   pos_ratio=0.930


  Movies_and_TV      positives= 5,713,713   explicit_neg=   773,077   pos_ratio=0.881
  CDs_and_Vinyl      positives= 1,284,065   explicit_neg=    86,087   pos_ratio=0.937
  Video_Games        positives=   604,377   explicit_neg=    77,663   pos_ratio=0.886


  Toys_and_Games     positives= 3,097,522   explicit_neg=   263,464   pos_ratio=0.922


### 3.3(b) Seen sets — from the full pre-5-core ratings

Seen sets span every interaction the user ever had in that vertical (including
the dropped 3s and items that didn't survive 5-core). Built once per vertical
from `RAW_RATINGS[vert]`, then the raw ratings table is freed.

In [11]:
def build_seen(df_all: pd.DataFrame, vertical: str) -> Dict[str, set]:
    seen: Dict[str, set] = defaultdict(set)
    for u, a in tqdm(zip(df_all['user_id'], df_all['parent_asin']),
                     total=len(df_all), desc=f'seen {vertical}'):
        seen[u].add(a)
    return dict(seen)


for vert in CFG.VERTICALS:
    SEEN[vert] = build_seen(RAW_RATINGS[vert], vert)
    print(f'  {vert:18s} seen users={len(SEEN[vert]):>9,d}   '
          f'mean items/user={np.mean([len(s) for s in SEEN[vert].values()]):.2f}')

# free the heavy raw frames — seen sets are all we needed from them
RAW_RATINGS.clear()


seen Books:   0%|          | 0/27106637 [00:00<?, ?it/s]

  Books              seen users=9,970,278   mean items/user=2.72


seen Movies_and_TV:   0%|          | 0/15910049 [00:00<?, ?it/s]

  Movies_and_TV      seen users=6,298,244   mean items/user=2.53


seen CDs_and_Vinyl:   0%|          | 0/4491733 [00:00<?, ?it/s]

  CDs_and_Vinyl      seen users=1,702,366   mean items/user=2.64


seen Video_Games:   0%|          | 0/4220244 [00:00<?, ?it/s]

  Video_Games        seen users=2,628,446   mean items/user=1.61


seen Toys_and_Games:   0%|          | 0/14961713 [00:00<?, ?it/s]

  Toys_and_Games     seen users=7,742,710   mean items/user=1.93


### 3.3(c) Sampled negatives — generated records

In [12]:
def sample_negatives(positives: pd.DataFrame, seen: Dict[str, set],
                     all_items: List[str], k: int, seed: int, vertical: str
                     ) -> pd.DataFrame:
    """For each positive (u, i, t), draw k items the user has not interacted
    with. Uniform over the unobserved catalog, seeded for reproducibility."""
    rng = np.random.default_rng(seed)
    item_arr = np.array(all_items)
    cat_size = len(item_arr)
    rows = []
    pos_users = positives['user_id'].values
    pos_ts    = positives['timestamp'].values
    for u, t_ in tqdm(zip(pos_users, pos_ts),
                      total=len(positives), desc=f'sample_neg {vertical}'):
        forbidden = seen.get(u, set())
        drawn = []
        tries = 0
        while len(drawn) < k and tries < k * 8:
            cand = item_arr[rng.integers(0, cat_size, size=k * 2)]
            for c in cand:
                if c not in forbidden and c not in drawn:
                    drawn.append(c)
                    if len(drawn) == k:
                        break
            tries += 1
        for c in drawn:
            rows.append((u, c, t_))
    out = pd.DataFrame(rows, columns=['user_id', 'parent_asin', 'timestamp'])
    out['rating'] = 0          # sentinel
    out['label']  = 0
    out['weight'] = 1.0
    return out


def combine_signals(labelled: pd.DataFrame, sampled_neg: pd.DataFrame) -> pd.DataFrame:
    keep = ['user_id', 'parent_asin', 'rating', 'timestamp', 'label', 'weight']
    return pd.concat([labelled[keep], sampled_neg[keep]], ignore_index=True)


for v_idx, vert in enumerate(CFG.VERTICALS):
    t0 = time.time()
    catalog  = ITEMS[vert]['parent_asin'].tolist()
    positives = LABELS[vert][LABELS[vert]['label'] == 1]
    NEG[vert] = sample_negatives(positives, SEEN[vert], catalog,
                                 CFG.neg_sample_ratio,
                                 seed=CFG.seed + v_idx,  # distinct seed per vertical
                                 vertical=vert)
    SIGNAL[vert] = combine_signals(LABELS[vert], NEG[vert])
    print(f'  {vert:18s} sampled_neg={len(NEG[vert]):>11,d}   '
          f'signal_rows={len(SIGNAL[vert]):>11,d}   '
          f'({time.time()-t0:.1f}s)')
    CONSTRUCT_STATS['by_vertical'][vert]['label_distribution']['sampled_negatives'] = int(len(NEG[vert]))


sample_neg Books:   0%|          | 0/7563762 [00:00<?, ?it/s]

  Books              sampled_neg= 30,255,048   signal_rows= 38,385,389   (61.9s)


sample_neg Movies_and_TV:   0%|          | 0/5713713 [00:00<?, ?it/s]

  Movies_and_TV      sampled_neg= 22,854,852   signal_rows= 29,341,642   (43.8s)


sample_neg CDs_and_Vinyl:   0%|          | 0/1284065 [00:00<?, ?it/s]

  CDs_and_Vinyl      sampled_neg=  5,136,260   signal_rows=  6,506,412   (9.8s)


sample_neg Video_Games:   0%|          | 0/604377 [00:00<?, ?it/s]

  Video_Games        sampled_neg=  2,417,508   signal_rows=  3,099,548   (4.1s)


sample_neg Toys_and_Games:   0%|          | 0/3097522 [00:00<?, ?it/s]

  Toys_and_Games     sampled_neg= 12,390,088   signal_rows= 15,751,074   (23.1s)


### 3.3(d) Item text embeddings — `all-MiniLM-L6-v2`, cached

The cache (`data/embed_cache/sentence-transformers_all-MiniLM-L6-v2.npz`)
is keyed by SHA1(text) and shared across **all 5 verticals** — cross-vertical
duplicate texts encode exactly once.

`embed_sample_size` caps the **new** texts encoded per vertical this run
(items the cache doesn't already cover). Anything beyond the cap receives
the placeholder zero embedding. Flip to `0` for a full encoding pass.

In [13]:
def pick_device(want: str) -> str:
    if want != 'auto':
        return want
    try:
        import torch
        if torch.backends.mps.is_available():
            return 'mps'
        if torch.cuda.is_available():
            return 'cuda'
    except ImportError:
        pass
    return 'cpu'


def text_hash(s: str) -> str:
    return hashlib.sha1(s.encode('utf-8')).hexdigest()


def _load_cache() -> Tuple[Path, Dict[str, np.ndarray]]:
    path = EMBED_CACHE / f'{CFG.embed_model.replace("/", "_")}.npz'
    if path.exists():
        z = np.load(path)
        return path, {k: z[k] for k in z.files}
    return path, {}


def encode_items(items: pd.DataFrame, vertical: str,
                 cache: Dict[str, np.ndarray], cache_path: Path,
                 model=None) -> Tuple[np.ndarray, Dict, object]:
    """Encode `items.item_text` (when present). Returns (emb, stats, model)
    where the model handle is passed back so subsequent verticals reuse it."""
    texts    = items['item_text'].tolist()
    has_text = items['has_text'].values
    hashes   = [text_hash(t) if t else '' for t in texts]

    cache_hits = 0
    need_idx, need_text = [], []
    for i, (t, h) in enumerate(zip(texts, hashes)):
        if not t or not has_text[i]:
            continue
        if h in cache:
            cache_hits += 1
            continue
        need_idx.append(i)
        need_text.append(t)

    sample_cap = CFG.embed_sample_size
    if sample_cap and len(need_text) > sample_cap:
        print(f'  {vertical}: smoke cap — encoding {sample_cap:,} of {len(need_text):,} new texts')
        need_idx  = need_idx[:sample_cap]
        need_text = need_text[:sample_cap]

    n_truncated = 0
    if need_text:
        from sentence_transformers import SentenceTransformer
        if model is None:
            device = pick_device(CFG.embed_device)
            print(f'  loading {CFG.embed_model} on device={device} (max_seq_length={CFG.max_seq_length})')
            model = SentenceTransformer(CFG.embed_model, device=device)
            model.max_seq_length = CFG.max_seq_length
        tok = model.tokenizer
        for t_ in need_text:
            if len(tok.encode(t_, add_special_tokens=False)) > CFG.max_seq_length:
                n_truncated += 1
        new_emb = model.encode(
            need_text, batch_size=CFG.embed_batch, show_progress_bar=True,
            convert_to_numpy=True, normalize_embeddings=CFG.embed_normalize,
        )
        for k, i in enumerate(need_idx):
            cache[hashes[i]] = new_emb[k]
        np.savez(cache_path, **cache)

    emb = np.zeros((len(items), CFG.embed_dim), dtype='float32')
    n_placeholder = 0
    for i, h in enumerate(hashes):
        if h and h in cache:
            emb[i] = cache[h]
        else:
            n_placeholder += 1
    stats = {
        'items_total':              int(len(emb)),
        'items_embedded':           int(len(emb) - n_placeholder),
        'items_placeholder':        int(n_placeholder),
        'cache_hits':               int(cache_hits),
        'texts_truncated_this_run': int(n_truncated),
    }
    print(f'  {vertical:18s} embedded={stats["items_embedded"]:>9,d}   '
          f'placeholder={stats["items_placeholder"]:>9,d}   '
          f'cache_hits={stats["cache_hits"]:>9,d}   '
          f'truncated_new={stats["texts_truncated_this_run"]:>6,d}')
    return emb, stats, model


CACHE_PATH, CACHE = _load_cache()
print(f'loaded {len(CACHE):,} cached vectors from {CACHE_PATH.name}')

_st_model = None
for vert in CFG.VERTICALS:
    t0 = time.time()
    EMB[vert], emb_stats, _st_model = encode_items(
        ITEMS[vert], vert, CACHE, CACHE_PATH, model=_st_model)
    print(f'    elapsed={time.time()-t0:.1f}s   cache_size={len(CACHE):,}')
    CONSTRUCT_STATS['by_vertical'][vert]['embedding'] = emb_stats
del _st_model


loaded 9,963 cached vectors from sentence-transformers_all-MiniLM-L6-v2.npz


  Books: smoke cap — encoding 2,000 of 435,094 new texts


  loading sentence-transformers/all-MiniLM-L6-v2 on device=mps (max_seq_length=256)


Token indices sequence length is longer than the specified maximum sequence length for this model (1022 > 256). Running this sequence through the model will result in indexing errors


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

  Video_Games        embedded=    2,024   placeholder=   20,722   cache_hits=        0   truncated_new=   999
    elapsed=10.1s   cache_size=17,957


  Toys_and_Games: smoke cap — encoding 2,000 of 148,572 new texts


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

  Toys_and_Games     embedded=    2,015   placeholder=  146,557   cache_hits=        0   truncated_new=   969
    elapsed=10.2s   cache_size=19,957


### 3.3(e) Category multi-hot — vocab shared across all 5 verticals

A single vocabulary is built across every item table so any vertical-to-
vertical comparison sits in the same feature space.

In [14]:
def build_category_vocab(item_dfs: Dict[str, pd.DataFrame]) -> Dict[str, int]:
    vocab: Dict[str, int] = {}
    for df in item_dfs.values():
        for cats in df['categories']:
            for c in cats:
                if c and c not in vocab:
                    vocab[c] = len(vocab)
    return vocab


def items_to_multihot(items: pd.DataFrame, vocab: Dict[str, int]) -> csr_matrix:
    rows, cols = [], []
    for i, cats in enumerate(items['categories']):
        for c in cats:
            j = vocab.get(c)
            if j is not None:
                rows.append(i); cols.append(j)
    data = np.ones(len(rows), dtype='float32')
    return csr_matrix((data, (rows, cols)), shape=(len(items), len(vocab)))


CAT_VOCAB = build_category_vocab(ITEMS)
print(f'global category vocab size: {len(CAT_VOCAB):,}')

for vert in CFG.VERTICALS:
    CAT[vert] = items_to_multihot(ITEMS[vert], CAT_VOCAB)
    paths = len({tuple(p) for p in ITEMS[vert]['categories']})
    print(f'  {vert:18s} multi-hot={CAT[vert].shape}   nnz={CAT[vert].nnz:>9,d}   '
          f'distinct_paths={paths:>6,d}')
    CONSTRUCT_STATS['by_vertical'][vert]['category_distinct_paths'] = int(paths)
CONSTRUCT_STATS['category_vocab_size'] = int(len(CAT_VOCAB))


global category vocab size: 3,092


  Books              multi-hot=(440134, 3092)   nnz=1,296,424   distinct_paths= 1,216
  Movies_and_TV      multi-hot=(181532, 3092)   nnz=  356,508   distinct_paths= 5,203
  CDs_and_Vinyl      multi-hot=(80990, 3092)   nnz=  265,505   distinct_paths=   690


  Video_Games        multi-hot=(22746, 3092)   nnz=   95,523   distinct_paths=   436
  Toys_and_Games     multi-hot=(148572, 3092)   nnz=  523,925   distinct_paths=   861


### 3.3(f) Popularity priors — `average_rating` + log1p(`rating_number`) z-scored

`rating_number` is heavy-tailed (most items have few ratings; a long tail has
millions). Per the spec, log1p + standardise per vertical so it doesn't
dominate downstream models. The z-score mean/std are recorded so they can be
re-applied to held-out items at inference.

In [15]:
for vert in CFG.VERTICALS:
    it = ITEMS[vert]
    rn = it['rating_number'].fillna(0).astype('float32')
    rn_log = np.log1p(rn)
    mu, sd = float(rn_log.mean()), float(rn_log.std() or 1.0)
    it['rating_number_z'] = ((rn_log - mu) / sd).astype('float32')
    it['avg_rating_filled'] = it['average_rating'].fillna(it['average_rating'].median()).astype('float32')
    POP[vert] = it[['parent_asin', 'avg_rating_filled', 'rating_number_z']].copy()
    CONSTRUCT_STATS['by_vertical'][vert]['popularity_priors'] = {
        'rating_number_log_mean': mu,
        'rating_number_log_std':  sd,
        'avg_rating_median':      float(it['average_rating'].median(skipna=True) or 0.0),
    }
    print(f'  {vert:18s} log1p(rating_number) mean={mu:.3f}  std={sd:.3f}')


  Books              log1p(rating_number) mean=5.565  std=1.720
  Movies_and_TV      log1p(rating_number) mean=6.123  std=1.912
  CDs_and_Vinyl      log1p(rating_number) mean=4.835  std=1.312
  Video_Games        log1p(rating_number) mean=5.338  std=1.545
  Toys_and_Games     log1p(rating_number) mean=5.446  std=1.441


## §3.4 Integrate Data — *Merged Data*

Two passes:

1. **Within-vertical join (per-vertical, all 5).** Join the labelled signal
   with the item-feature table on `parent_asin`. §2.2.8 guarantees zero
   orphans; we assert it. After this pass each vertical has an enriched
   interaction table ready for ID-remapping in §3.5.
2. **Cross-vertical user bridge (loop over the 8 pairs).** For each
   `(SOURCE, TARGET)`: intersect the two 5-core user sets, log overlap +
   Jaccard, check the floor, build matched (source, target) training
   pairs and the role splits (source-only / target-only / overlap).

### Within-vertical join

In [16]:
def user_activity(signal: pd.DataFrame, users: Set[str], side: str) -> pd.DataFrame:
    pos = signal[(signal['label'] == 1) & (signal['user_id'].isin(users))]
    g = pos.groupby('user_id').agg(
        pos_count=('parent_asin', 'count'),
        first_ts=('timestamp', 'min'),
        last_ts=('timestamp', 'max'),
    )
    g.columns = [f'{side}_{c}' for c in g.columns]
    return g


for vert in CFG.VERTICALS:
    item_set = set(ITEMS[vert]['parent_asin'])
    cov = float(RATINGS_5CORE[vert]['parent_asin'].isin(item_set).mean())
    # join the SIGNAL (positives + explicit negs + sampled negs) to verify
    # every signal row has metadata — sampled negs may reference items dropped
    # by 5-core; we'll remove those at remap time.
    sig_cov = float(SIGNAL[vert]['parent_asin'].isin(item_set).mean())
    INTEGRATE_STATS[vert] = {
        'ratings_meta_coverage_pct': round(cov * 100, 6),
        'signal_meta_coverage_pct':  round(sig_cov * 100, 6),
        'signal_rows':               int(len(SIGNAL[vert])),
        'unique_users':              int(SIGNAL[vert]['user_id'].nunique()),
    }
    print(f'  {vert:18s} ratings⨝meta={cov*100:.4f}%   signal⨝meta={sig_cov*100:.4f}%   '
          f'rows={len(SIGNAL[vert]):>11,d}')
    assert cov == 1.0, f'{vert}: ratings have orphans (cov={cov})'

log_stage('3.4_integrate_within', **INTEGRATE_STATS)


  Books              ratings⨝meta=100.0000%   signal⨝meta=100.0000%   rows= 38,385,389


  Movies_and_TV      ratings⨝meta=100.0000%   signal⨝meta=100.0000%   rows= 29,341,642


  CDs_and_Vinyl      ratings⨝meta=100.0000%   signal⨝meta=100.0000%   rows=  6,506,412


  Video_Games        ratings⨝meta=100.0000%   signal⨝meta=100.0000%   rows=  3,099,548


  Toys_and_Games     ratings⨝meta=100.0000%   signal⨝meta=100.0000%   rows= 15,751,074
[3.4_integrate_within]
  Books:
    ratings_meta_coverage_pct: 100.0
    signal_meta_coverage_pct: 100.0
    signal_rows: 38385389
    unique_users: 687023
  Movies_and_TV:
    ratings_meta_coverage_pct: 100.0
    signal_meta_coverage_pct: 100.0
    signal_rows: 29341642
    unique_users: 592843
  CDs_and_Vinyl:
    ratings_meta_coverage_pct: 100.0
    signal_meta_coverage_pct: 100.0
    signal_rows: 6506412
    unique_users: 112342
  Video_Games:
    ratings_meta_coverage_pct: 100.0
    signal_meta_coverage_pct: 100.0
    signal_rows: 3099548
    unique_users: 80886
  Toys_and_Games:
    ratings_meta_coverage_pct: 100.0
    signal_meta_coverage_pct: 100.0
    signal_rows: 15751074
    unique_users: 379869


### Cross-vertical user bridge — 8 pairs

In [17]:
# canonical pair set (optionally extended with reverse directions)
PAIR_SET = list(CFG.PAIRS)
if CFG.ADD_REVERSE:
    PAIR_SET = PAIR_SET + [(t, s) for s, t in CFG.PAIRS]

USERS_5CORE: Dict[str, Set[str]] = {
    v: set(RATINGS_5CORE[v]['user_id'].unique()) for v in CFG.VERTICALS
}

for src, tgt in PAIR_SET:
    s_users, t_users = USERS_5CORE[src], USERS_5CORE[tgt]
    shared = s_users & t_users
    union  = s_users | t_users
    jacc = len(shared) / max(1, len(union))
    borderline = len(shared) < CFG.OVERLAP_FLOOR
    if borderline:
        warnings.warn(f'{src}->{tgt}: shared={len(shared):,} below floor '
                      f'{CFG.OVERLAP_FLOOR:,} — R1 contingency')

    # role splits (sets of user_id strings)
    src_only = s_users - t_users
    tgt_only = t_users - s_users

    # per-side activity for the shared users
    src_act = user_activity(SIGNAL[src], shared, 'src')
    tgt_act = user_activity(SIGNAL[tgt], shared, 'tgt')
    shared_df = src_act.join(tgt_act, how='inner').reset_index()

    key = f'{src}__{tgt}'
    PAIR_STATS[key] = {
        'source':         src,
        'target':         tgt,
        'shared_users':   int(len(shared)),
        'jaccard':        round(jacc, 6),
        'borderline':     bool(borderline),
        'src_only_users': int(len(src_only)),
        'tgt_only_users': int(len(tgt_only)),
        'shared_active_both_sides': int(len(shared_df)),
        '_shared_user_ids':  shared,    # kept in memory; not serialised here
        '_shared_df':        shared_df, # ditto
    }
    print(f'  {src:>15s} -> {tgt:<15s} '
          f'shared={len(shared):>7,d}   jacc={jacc:.4f}   '
          f'src_only={len(src_only):>8,d}   tgt_only={len(tgt_only):>8,d}'
          f'{"   [BORDERLINE]" if borderline else ""}')

log_stage('3.4_integrate_pairs',
          **{k: {kk: vv for kk, vv in v.items() if not kk.startswith('_')}
             for k, v in PAIR_STATS.items()})


            Books -> Movies_and_TV   shared=109,206   jacc=0.0933   src_only= 577,817   tgt_only= 483,637


            Books -> Toys_and_Games  shared= 75,649   jacc=0.0763   src_only= 611,374   tgt_only= 304,220


    Movies_and_TV -> Toys_and_Games  shared= 46,024   jacc=0.0497   src_only= 546,819   tgt_only= 333,845


    Movies_and_TV -> CDs_and_Vinyl   shared= 34,630   jacc=0.0516   src_only= 558,213   tgt_only=  77,712


            Books -> CDs_and_Vinyl   shared= 27,570   jacc=0.0357   src_only= 659,453   tgt_only=  84,772


    Movies_and_TV -> Video_Games     shared= 16,504   jacc=0.0251   src_only= 576,339   tgt_only=  64,382


   Toys_and_Games -> Video_Games     shared= 14,517   jacc=0.0325   src_only= 365,352   tgt_only=  66,369


            Books -> Video_Games     shared= 11,801   jacc=0.0156   src_only= 675,222   tgt_only=  69,085
[3.4_integrate_pairs]
  Books__Movies_and_TV:
    source: Books
    target: Movies_and_TV
    shared_users: 109206
    jaccard: 0.093286
    borderline: False
    src_only_users: 577817
    tgt_only_users: 483637
    shared_active_both_sides: 108936
  Books__Toys_and_Games:
    source: Books
    target: Toys_and_Games
    shared_users: 75649
    jaccard: 0.076317
    borderline: False
    src_only_users: 611374
    tgt_only_users: 304220
    shared_active_both_sides: 75599
  Movies_and_TV__Toys_and_Games:
    source: Movies_and_TV
    target: Toys_and_Games
    shared_users: 46024
    jaccard: 0.049665
    borderline: False
    src_only_users: 546819
    tgt_only_users: 333845
    shared_active_both_sides: 45954
  Movies_and_TV__CDs_and_Vinyl:
    source: Movies_and_TV
    target: CDs_and_Vinyl
    shared_users: 34630
    jaccard: 0.051644
    borderline: False
    src_only_users: 

## §3.5 Format Data — *Reformatted Data*

Per-vertical (Stage A output): temporal train/val/test split → ID remap
(per-vertical contiguous integers) → schema standardisation → Parquet + NPZ
writes to `data/processed/{vertical}/`.

Per-pair (Stage B output): global `user_id → user_idx` map across the union
of both verticals' users (so a shared user gets the same index on both
sides) → re-emit interactions with the global user index → write
`data/processed/pairs/{SRC}__{TGT}/`.

### Stage A — per-vertical writes

In [18]:
def temporal_split(df: pd.DataFrame,
                   ratios: Tuple[float, float, float]) -> Tuple[pd.Series, Dict]:
    n = len(df)
    train_end = int(ratios[0] * n)
    val_end   = int((ratios[0] + ratios[1]) * n)
    order = np.argsort(df['timestamp'].values, kind='mergesort')
    sorted_ts = df['timestamp'].values[order]
    out = np.array(['train'] * n, dtype=object)
    out[order[train_end:val_end]] = 'val'
    out[order[val_end:]]          = 'test'
    cutoffs = {
        'train_end_ts_ms': int(sorted_ts[train_end - 1]) if train_end > 0 else None,
        'val_end_ts_ms':   int(sorted_ts[val_end   - 1]) if val_end   > 0 else None,
        'train_end_date':  str(pd.to_datetime(sorted_ts[train_end - 1], unit='ms').date())
                           if train_end > 0 else None,
        'val_end_date':    str(pd.to_datetime(sorted_ts[val_end   - 1], unit='ms').date())
                           if val_end   > 0 else None,
    }
    return pd.Series(out, index=df.index, name='split'), cutoffs


def write_vertical(vert: str) -> Dict:
    out_dir = PROCESSED / vert
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / 'id_maps').mkdir(exist_ok=True)

    items = ITEMS[vert]
    asins = items['parent_asin'].tolist()
    item2idx = {a: i for i, a in enumerate(asins)}

    # per-vertical user vocab (Stage A baseline). Stage B reindexes to a pair-global map.
    users = sorted(set(SIGNAL[vert]['user_id']))
    user2idx = {u: i for i, u in enumerate(users)}

    sig = SIGNAL[vert].copy()
    sig['user_idx'] = sig['user_id'].map(user2idx).astype('int64')
    sig['item_idx'] = sig['parent_asin'].map(item2idx)
    n_drop = int(sig['item_idx'].isna().sum())
    if n_drop:
        sig = sig.dropna(subset=['item_idx'])
    sig['item_idx'] = sig['item_idx'].astype('int64')

    split_series, cutoffs = temporal_split(sig, CFG.split_ratios)
    sig['split'] = split_series.values

    inter_cols = ['user_idx', 'item_idx', 'label', 'weight', 'rating', 'timestamp', 'split']
    sig[inter_cols].to_parquet(out_dir / 'interactions.parquet', index=False)

    # popularity priors aligned to item_idx order
    pop = POP[vert].set_index('parent_asin').reindex(asins).reset_index()

    np.savez(
        out_dir / 'item_features.npz',
        text_emb=EMB[vert],
        has_text=items['has_text'].values.astype('bool'),
        text_source=np.array(items['text_source'].tolist(), dtype=object),
        cat_indptr=CAT[vert].indptr,
        cat_indices=CAT[vert].indices,
        cat_shape=np.array(CAT[vert].shape),
        avg_rating=pop['avg_rating_filled'].values.astype('float32'),
        rating_number_z=pop['rating_number_z'].values.astype('float32'),
        item_idx_to_asin=np.array(asins, dtype=object),
    )

    # seen sets — long form, useful for inference-time exclusion
    rows = []
    for u, items_set in SEEN[vert].items():
        ui = user2idx.get(u)
        if ui is None:
            continue
        for it in items_set:
            ii = item2idx.get(it)
            if ii is not None:
                rows.append((ui, ii))
    pd.DataFrame(rows, columns=['user_idx', 'item_idx']
                 ).to_parquet(out_dir / 'seen.parquet', index=False)

    # id maps
    pd.DataFrame({'user_id': list(user2idx.keys()),
                  'user_idx': list(user2idx.values())}
                 ).to_parquet(out_dir / 'id_maps' / 'user.parquet', index=False)
    pd.DataFrame({'parent_asin': asins,
                  'item_idx': range(len(asins))}
                 ).to_parquet(out_dir / 'id_maps' / 'item.parquet', index=False)

    split_counts = {k: int(v) for k, v in sig['split'].value_counts().items()}
    return {
        'rows':           int(len(sig)),
        'rows_dropped_no_meta': n_drop,
        'n_users':        int(len(user2idx)),
        'n_items':        int(len(item2idx)),
        'split_counts':   split_counts,
        'cutoffs':        cutoffs,
        'paths':          [str(p.relative_to(PROCESSED)) for p in sorted(out_dir.rglob('*')) if p.is_file()],
    }


for vert in CFG.VERTICALS:
    t0 = time.time()
    FORMAT_STATS[vert] = write_vertical(vert)
    print(f'  {vert:18s} rows={FORMAT_STATS[vert]["rows"]:>11,d}   '
          f'users={FORMAT_STATS[vert]["n_users"]:>9,d}   '
          f'items={FORMAT_STATS[vert]["n_items"]:>9,d}   '
          f'splits={FORMAT_STATS[vert]["split_counts"]}   '
          f'({time.time()-t0:.1f}s)')


  Books              rows= 38,385,389   users=  687,023   items=  440,134   splits={'train': 30708311, 'val': 3838539, 'test': 3838539}   (58.9s)


  Movies_and_TV      rows= 29,341,642   users=  592,843   items=  181,532   splits={'train': 23473313, 'test': 2934165, 'val': 2934164}   (35.7s)


  CDs_and_Vinyl      rows=  6,506,412   users=  112,342   items=   80,990   splits={'train': 5205129, 'test': 650642, 'val': 650641}   (7.7s)


  Video_Games        rows=  3,099,548   users=   80,886   items=   22,746   splits={'train': 2479638, 'val': 309955, 'test': 309955}   (4.7s)


  Toys_and_Games     rows= 15,751,074   users=  379,869   items=  148,572   splits={'train': 12600859, 'test': 1575108, 'val': 1575107}   (22.8s)


### Stage B — per-pair writes

In [19]:
def write_pair(src: str, tgt: str) -> Dict:
    key = f'{src}__{tgt}'
    stats = PAIR_STATS[key]
    shared = stats['_shared_user_ids']
    shared_df = stats['_shared_df']

    out_dir = PAIRS_DIR / key
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / 'id_maps').mkdir(exist_ok=True)

    # pair-global user vocab — union of both verticals' signal users.
    users = sorted(set(SIGNAL[src]['user_id']).union(SIGNAL[tgt]['user_id']))
    user2idx_pair = {u: i for i, u in enumerate(users)}

    def reindex(vert: str) -> pd.DataFrame:
        items = ITEMS[vert]['parent_asin'].tolist()
        item2idx = {a: i for i, a in enumerate(items)}
        sig = SIGNAL[vert].copy()
        sig['user_idx'] = sig['user_id'].map(user2idx_pair).astype('int64')
        sig['item_idx'] = sig['parent_asin'].map(item2idx)
        sig = sig.dropna(subset=['item_idx'])
        sig['item_idx'] = sig['item_idx'].astype('int64')
        split_series, _ = temporal_split(sig, CFG.split_ratios)
        sig['split'] = split_series.values
        return sig[['user_idx', 'item_idx', 'label', 'weight',
                    'rating', 'timestamp', 'split']]

    src_inter = reindex(src)
    tgt_inter = reindex(tgt)
    src_inter.to_parquet(out_dir / 'source_interactions.parquet', index=False)
    tgt_inter.to_parquet(out_dir / 'target_interactions.parquet', index=False)

    # shared users frame, with pair-global user_idx
    sdf = shared_df.copy()
    sdf['user_idx'] = sdf['user_id'].map(user2idx_pair).astype('int64')
    sdf = sdf[['user_idx', 'src_pos_count', 'tgt_pos_count',
               'src_first_ts', 'src_last_ts', 'tgt_first_ts', 'tgt_last_ts']]
    sdf.to_parquet(out_dir / 'shared_users.parquet', index=False)

    # matched_pairs.parquet — one row per shared user; the actual source-vec
    # aggregator is left to §4 (default = mean-pool the user's source-item
    # embeddings). We persist the user_idx + counts; the modelling layer pulls
    # the embedding rows from data/processed/<source>/item_features.npz.
    matched = sdf[['user_idx', 'src_pos_count', 'tgt_pos_count']].copy()
    matched.to_parquet(out_dir / 'matched_pairs.parquet', index=False)

    # role splits
    src_only = sorted(set(SIGNAL[src]['user_id']) - set(SIGNAL[tgt]['user_id']))
    tgt_only = sorted(set(SIGNAL[tgt]['user_id']) - set(SIGNAL[src]['user_id']))
    roles_df = pd.DataFrame({
        'user_idx': [user2idx_pair[u] for u in src_only] +
                    [user2idx_pair[u] for u in tgt_only] +
                    [user2idx_pair[u] for u in sorted(shared)],
        'role':     (['source_only'] * len(src_only) +
                     ['target_only'] * len(tgt_only) +
                     ['overlap']     * len(shared)),
    })
    roles_df.to_parquet(out_dir / 'roles.parquet', index=False)

    # id maps
    pd.DataFrame({'user_id':  list(user2idx_pair.keys()),
                  'user_idx': list(user2idx_pair.values())}
                 ).to_parquet(out_dir / 'id_maps' / 'user.parquet', index=False)

    pair_meta = {
        'config':           asdict(CFG),
        'source':           src,
        'target':           tgt,
        'overlap':          int(len(shared)),
        'jaccard':          stats['jaccard'],
        'overlap_floor':    CFG.OVERLAP_FLOOR,
        'borderline':       stats['borderline'],
        'profile_aggregator': 'mean_pool_source_embeddings',
        'snapshot_date':    time.strftime('%Y-%m-%d'),
    }
    with open(out_dir / 'meta.json', 'w') as fh:
        json.dump(pair_meta, fh, indent=2, default=str)

    n_files = sum(1 for _ in out_dir.rglob('*') if _.is_file())
    return {
        **{k: v for k, v in stats.items() if not k.startswith('_')},
        'n_files_written': n_files,
        'pair_global_user_vocab': int(len(user2idx_pair)),
    }


for src, tgt in PAIR_SET:
    t0 = time.time()
    res = write_pair(src, tgt)
    PAIR_STATS[f'{src}__{tgt}'].update(res)
    print(f'  {src:>15s} -> {tgt:<15s} '
          f'overlap={res["shared_users"]:>7,d}   '
          f'pair_users={res["pair_global_user_vocab"]:>9,d}   '
          f'files={res["n_files_written"]}   '
          f'({time.time()-t0:.1f}s)')


            Books -> Movies_and_TV   overlap=109,206   pair_users=1,170,660   files=7   (65.8s)


            Books -> Toys_and_Games  overlap= 75,649   pair_users=  991,243   files=7   (51.0s)


    Movies_and_TV -> Toys_and_Games  overlap= 46,024   pair_users=  926,688   files=7   (39.9s)


    Movies_and_TV -> CDs_and_Vinyl   overlap= 34,630   pair_users=  670,555   files=7   (30.2s)


            Books -> CDs_and_Vinyl   overlap= 27,570   pair_users=  771,795   files=7   (42.2s)


    Movies_and_TV -> Video_Games     overlap= 16,504   pair_users=  657,225   files=7   (29.7s)


   Toys_and_Games -> Video_Games     overlap= 14,517   pair_users=  446,238   files=7   (14.9s)


            Books -> Video_Games     overlap= 11,801   pair_users=  756,108   files=7   (41.4s)


## Stage C — Dataset Description

The §3 phase output. Consolidates everything Stage A and Stage B produced
into a single human-readable file (`DATASET_DESCRIPTION.md`) plus the JSON
mirror (`dataset_description.json`) for downstream code.

In [20]:
def fmt(n):
    if isinstance(n, float):
        return f'{n:,.4f}' if abs(n) < 1 else f'{n:,.2f}'
    if isinstance(n, int):
        return f'{n:,d}'
    return str(n)


SNAPSHOT = time.strftime('%Y-%m-%d')

dataset = {
    'snapshot_date':   SNAPSHOT,
    'config':          asdict(CFG),
    'category_vocab_size': CONSTRUCT_STATS['category_vocab_size'],
    'verticals': {
        v: {
            'select':    SELECT_STATS[v],
            'clean':     CLEAN_STATS[v],
            'construct': CONSTRUCT_STATS['by_vertical'][v],
            'integrate': INTEGRATE_STATS[v],
            'format':    FORMAT_STATS[v],
        } for v in CFG.VERTICALS
    },
    'pairs': {
        k: {kk: vv for kk, vv in v.items() if not kk.startswith('_')}
        for k, v in PAIR_STATS.items()
    },
}
with open(PROCESSED / 'dataset_description.json', 'w') as fh:
    json.dump(dataset, fh, indent=2, default=str)
print(f'wrote {(PROCESSED / "dataset_description.json").relative_to(PROJECT_ROOT)}')


wrote data/processed/dataset_description.json


In [21]:
lines: List[str] = []
P = lines.append

P('# Dataset Description — CRISP-DM §3 Phase Output')
P('')
P(f'_Snapshot: {SNAPSHOT}._  '
  f'Generated by `notebooks/data_prep.ipynb` from `section3_preprocessing_plan.md`.')
P('')
P(f'- **Verticals processed (Stage A):** {len(CFG.VERTICALS)} — {", ".join(CFG.VERTICALS)}')
P(f'- **Pairs bridged (Stage B):** {len(PAIR_SET)} '
  f'{"(includes reverse directions)" if CFG.ADD_REVERSE else ""}')
P(f'- **Overlap floor:** {CFG.OVERLAP_FLOOR:,} shared users (sub-floor pairs flagged as R1)')
P(f'- **Embedding model:** `{CFG.embed_model}` (dim={CFG.embed_dim}, '
  f'max_seq_length={CFG.max_seq_length}, L2-normalize={CFG.embed_normalize})')
P(f'- **Smoke mode:** `embed_sample_size={CFG.embed_sample_size}` '
  f'(`0` = full run)')
P(f'- **Split:** {CFG.split} {list(CFG.split_ratios)}')
P(f'- **Seed:** {CFG.seed}')
P(f'- **Global category vocab size:** {fmt(CONSTRUCT_STATS["category_vocab_size"])}')
P('')

# --- Per-vertical block ---
P('## §3.1 / §3.2 — Per-vertical selection + cleaning')
P('')
P('| vertical | 0-core rows | dropped 0 | dropped 3 | 5-core rows | 5-core users | 5-core items | retention % | k |')
P('|---|---:|---:|---:|---:|---:|---:|---:|---:|')
for v in CFG.VERTICALS:
    s = SELECT_STATS[v]
    P(f'| `{v}` | {fmt(s["interactions_0core"])} | {fmt(s["dropped_rating_0"])} | '
      f'{fmt(s["dropped_rating_3"])} | {fmt(s["interactions_5core"])} | '
      f'{fmt(s["users_5core"])} | {fmt(s["items_5core"])} | '
      f'{s["retention_pct_vs_0core"]:.2f} | {s["k_core"]} |')
P('')

P('### Item-text fallback chain (§3.2)')
P('')
P('| vertical | items kept | primary | fallback | placeholder | orphans | dup pairs |')
P('|---|---:|---:|---:|---:|---:|---:|')
for v in CFG.VERTICALS:
    c = CLEAN_STATS[v]
    P(f'| `{v}` | {fmt(c["items_kept"])} | {fmt(c["text_primary"])} | '
      f'{fmt(c["text_fallback"])} | {fmt(c["text_none_placeholder"])} | '
      f'{fmt(c["orphan_interactions"])} | {fmt(c["duplicate_pairs"])} |')
P('')

P('### Per-field missing-value rates on kept items')
P('')
P('| field | ' + ' | '.join(f'`{v}`' for v in CFG.VERTICALS) + ' |')
P('|---|' + '|'.join(['---:'] * len(CFG.VERTICALS)) + '|')
for fld in TRACKED_FIELDS:
    row = [f'`{fld}`']
    for v in CFG.VERTICALS:
        r = CLEAN_STATS[v]['missing_value_rates'].get(fld, 0)
        row.append(f'{r*100:.2f}%')
    P('| ' + ' | '.join(row) + ' |')
P('')

# --- §3.3 ---
P('## §3.3 — Construct: labels, embeddings, categories, popularity')
P('')
P('| vertical | positives | explicit_neg | dropped 3 | sampled_neg | items embedded | items placeholder | cache hits |')
P('|---|---:|---:|---:|---:|---:|---:|---:|')
for v in CFG.VERTICALS:
    ld = CONSTRUCT_STATS['by_vertical'][v]['label_distribution']
    em = CONSTRUCT_STATS['by_vertical'][v]['embedding']
    P(f'| `{v}` | {fmt(ld["positives"])} | {fmt(ld["explicit_negatives"])} | '
      f'{fmt(ld["dropped_3"])} | {fmt(ld["sampled_negatives"])} | '
      f'{fmt(em["items_embedded"])} | {fmt(em["items_placeholder"])} | '
      f'{fmt(em["cache_hits"])} |')
P('')

# --- §3.5 ---
P('## §3.5 — Format: per-vertical splits + ID vocabs')
P('')
P('| vertical | rows | users | items | train | val | test | train end | val end |')
P('|---|---:|---:|---:|---:|---:|---:|---|---|')
for v in CFG.VERTICALS:
    f = FORMAT_STATS[v]
    sc = f['split_counts']
    co = f['cutoffs']
    P(f'| `{v}` | {fmt(f["rows"])} | {fmt(f["n_users"])} | {fmt(f["n_items"])} | '
      f'{fmt(sc.get("train", 0))} | {fmt(sc.get("val", 0))} | {fmt(sc.get("test", 0))} | '
      f'{co.get("train_end_date")} | {co.get("val_end_date")} |')
P('')

# --- §3.4 pairs ---
P('## §3.4 — Cross-vertical bridge (8 pairs)')
P('')
P('| pair | shared users | jaccard | source_only | target_only | active both sides | borderline |')
P('|---|---:|---:|---:|---:|---:|:---:|')
for src, tgt in PAIR_SET:
    s = PAIR_STATS[f'{src}__{tgt}']
    P(f'| `{src} -> {tgt}` | {fmt(s["shared_users"])} | {s["jaccard"]:.4f} | '
      f'{fmt(s["src_only_users"])} | {fmt(s["tgt_only_users"])} | '
      f'{fmt(s["shared_active_both_sides"])} | '
      f'{"YES" if s["borderline"] else "no"} |')
P('')

# --- Schema ---
P('## Output schemas')
P('')
P('**Stage A — `data/processed/{vertical}/`**')
P('')
P('- `interactions.parquet` — `user_idx, item_idx, label, weight, rating, timestamp, split`')
P('- `item_features.npz` — `text_emb (n,384), has_text (n,), text_source (n,), '
  'cat_indptr / cat_indices / cat_shape (sparse multi-hot), '
  'avg_rating (n,), rating_number_z (n,), item_idx_to_asin (n,)`')
P('- `seen.parquet` — `user_idx, item_idx` (long form)')
P('- `id_maps/user.parquet`, `id_maps/item.parquet`')
P('')
P('**Stage B — `data/processed/pairs/{SRC}__{TGT}/`**')
P('')
P('- `source_interactions.parquet`, `target_interactions.parquet` — same schema as Stage A,'
  ' but with the pair-global `user_idx`')
P('- `shared_users.parquet` — `user_idx, src_pos_count, tgt_pos_count, '
  'src_first_ts, src_last_ts, tgt_first_ts, tgt_last_ts`')
P('- `matched_pairs.parquet` — `user_idx, src_pos_count, tgt_pos_count` (the EMCDR training rows)')
P('- `roles.parquet` — `user_idx, role ∈ {source_only, target_only, overlap}`')
P('- `id_maps/user.parquet`, `meta.json`')
P('')

with open(PROCESSED / 'DATASET_DESCRIPTION.md', 'w') as fh:
    fh.write('\n'.join(lines))
print(f'wrote {(PROCESSED / "DATASET_DESCRIPTION.md").relative_to(PROJECT_ROOT)}')

# Final artifact listing
print('\n=== ARTIFACTS ===')
total_mb = 0
for p in sorted(PROCESSED.rglob('*')):
    if p.is_file():
        sz_mb = p.stat().st_size / 1e6
        total_mb += sz_mb
        print(f'  {sz_mb:>9.2f} MB  {p.relative_to(PROCESSED)}')
print(f'  {"-"*60}')
print(f'  {total_mb:>9.2f} MB  total')


wrote data/processed/DATASET_DESCRIPTION.md

=== ARTIFACTS ===
       6.36 MB  Books/id_maps/item.parquet
      20.77 MB  Books/id_maps/user.parquet
     365.41 MB  Books/interactions.parquet
     693.56 MB  Books/item_features.npz
      47.55 MB  Books/seen.parquet
       1.32 MB  CDs_and_Vinyl/id_maps/item.parquet
       3.70 MB  CDs_and_Vinyl/id_maps/user.parquet
      44.59 MB  CDs_and_Vinyl/interactions.parquet
     127.73 MB  CDs_and_Vinyl/item_features.npz
       4.92 MB  CDs_and_Vinyl/seen.parquet
       0.01 MB  DATASET_DESCRIPTION.md
       2.86 MB  Movies_and_TV/id_maps/item.parquet
      18.00 MB  Movies_and_TV/id_maps/user.parquet
     263.60 MB  Movies_and_TV/interactions.parquet
     285.35 MB  Movies_and_TV/item_features.npz
      30.86 MB  Movies_and_TV/seen.parquet
       2.48 MB  Toys_and_Games/id_maps/item.parquet
      11.71 MB  Toys_and_Games/id_maps/user.parquet
     138.73 MB  Toys_and_Games/interactions.parquet
     234.47 MB  Toys_and_Games/item_features.npz
 